In [16]:
# =========================
# IMPORT THƯ VIỆN
# =========================

import pandas as pd
import numpy as np


# =========================
# ĐỌC FILE CSV
# =========================

df = pd.read_csv('dulieuxettuyendaihoc.csv')


# =========================
# IN 10 DÒNG ĐẦU
# =========================

print("10 dòng đầu:")
print(df.head(10))


# =========================
# IN 10 DÒNG CUỐI
# =========================

print("\n10 dòng cuối:")
print(df.tail(10))


# =========================
# PHÂN LOẠI DỮ LIỆU
# =========================

dinh_tinh = ['GT', 'DT', 'KV', 'KT']

dinh_luong = [col for col in df.columns if col not in dinh_tinh]

print("\nDữ liệu định tính:")
print(dinh_tinh)

print("\nDữ liệu định lượng:")
print(dinh_luong)


# =========================
# XỬ LÝ THIẾU DỮ LIỆU DT
# =========================

print("\nSố lượng thiếu DT:")
print(df['DT'].isnull().sum())

print("\nGiá trị riêng biệt DT:")
print(df['DT'].unique())

print("\nBảng tần số DT:")
print(df['DT'].value_counts(dropna=False))

# Điền giá trị 0
df['DT'] = df['DT'].fillna(0)


# =========================
# XỬ LÝ THIẾU T1 BẰNG MEAN
# =========================

print("\nSố lượng thiếu T1:")
print(df['T1'].isnull().sum())

mean_T1 = df['T1'].mean()

df['T1'] = df['T1'].fillna(mean_T1)

print("\nMean T1:")
print(mean_T1)


# =========================
# XỬ LÝ TẤT CẢ CỘT ĐIỂM
# =========================

cot_diem = [
    'T1','L1','H1','S1','V1','X1','D1','N1',
    'T2','L2','H2','S2','V2','X2','D2','N2',
    'T6','L6','H6','S6','V6','X6','D6','N6',
    'DH1','DH2','DH3'
]

for col in cot_diem:
    df[col] = df[col].fillna(df[col].mean())


# =========================
# TẠO TBM1 TBM2 TBM3
# =========================

df['TBM1'] = (
    df['T1']*2 + df['L1'] + df['H1'] + df['S1']
    + df['V1']*2 + df['X1'] + df['D1'] + df['N1']
) / 10

df['TBM2'] = (
    df['T2']*2 + df['L2'] + df['H2'] + df['S2']
    + df['V2']*2 + df['X2'] + df['D2'] + df['N2']
) / 10

df['TBM3'] = (
    df['T6']*2 + df['L6'] + df['H6'] + df['S6']
    + df['V6']*2 + df['X6'] + df['D6'] + df['N6']
) / 10


# =========================
# XẾP LOẠI HỌC LỰC
# =========================

def xep_loai(tbm):

    if tbm < 5:
        return 'Y'

    elif tbm < 6.5:
        return 'TB'

    elif tbm < 8:
        return 'K'

    elif tbm < 9:
        return 'G'

    else:
        return 'XS'


df['XL1'] = df['TBM1'].apply(xep_loai)
df['XL2'] = df['TBM2'].apply(xep_loai)
df['XL3'] = df['TBM3'].apply(xep_loai)


# =========================
# CHUYỂN THANG ĐIỂM 4
# =========================

def minmax4(col):

    return 4 * (col - col.min()) / (col.max() - col.min())


df['US_TBM1'] = minmax4(df['TBM1'])
df['US_TBM2'] = minmax4(df['TBM2'])
df['US_TBM3'] = minmax4(df['TBM3'])


# =========================
# TẠO KQXT
# =========================

def kqxt(row):

    if row['KT'] in ['A', 'A1']:

        diem = (
            row['DH1']*2 +
            row['DH2'] +
            row['DH3']
        ) / 4

    elif row['KT'] == 'B':

        diem = (
            row['DH1'] +
            row['DH2']*2 +
            row['DH3']
        ) / 4

    else:

        diem = (
            row['DH1'] +
            row['DH2'] +
            row['DH3']
        ) / 3

    if diem >= 5:
        return 1

    else:
        return 0


df['KQXT'] = df.apply(kqxt, axis=1)


# =========================
# LƯU FILE MỚI
# =========================

df.to_csv(
    'processed_dulieuxettuyendaihoc.csv',
    index=False
)

print("\nĐã lưu file processed_dulieuxettuyendaihoc.csv")


# =========================
# XEM 5 DÒNG CUỐI
# =========================

print(df.tail())



10 dòng đầu:
   STT   T1   L1   H1   S1   V1   X1   D1   N1   T2  ...   X6   D6   N6  GT  \
0    1  7.2  7.3  6.3  7.3  7.0  7.9  7.3  5.5  8.4  ...  6.6  7.6  5.9   F   
1    2  5.4  3.9  3.9  4.0  5.4  5.4  5.3  2.8  6.3  ...  6.6  6.1  4.4   M   
2    3  5.6  6.8  7.2  7.5  4.3  7.4  5.8  3.2  5.0  ...  7.9  8.1  4.6   M   
3    4  6.6  6.4  5.3  6.9  5.4  7.3  6.4  5.8  5.1  ...  7.1  7.3  7.4   M   
4    5  6.0  5.0  6.0  7.3  6.5  7.7  7.9  6.1  5.4  ...  6.1  7.5  7.2   M   
5    6  9.3  7.6  7.9  8.6  7.0  7.3  7.7  7.9  9.6  ...  5.7  8.0  7.8   M   
6    7  2.8  3.9  5.5  6.9  5.0  7.3  4.6  5.2  4.4  ...  6.6  6.0  6.0   F   
7    8  8.3  6.0  7.6  5.1  7.5  4.7  5.8  7.2  6.7  ...  7.1  6.8  7.0   F   
8    9  6.5  6.3  7.6  6.0  5.5  7.1  6.3  5.0  7.3  ...  9.1  7.9  6.1   F   
9   10  7.3  5.9  4.7  7.1  6.7  7.9  6.7  7.7  8.0  ...  6.4  6.1  7.8   F   

   DT   KV   DH1   DH2   DH3  KT  
0 NaN  2NT  3.25  3.25  4.50  A1  
1 NaN    1  6.00  4.00  3.50   C  
2 NaN    1  